# Procedural Readout Exploration

This notebook loads the saved study artifacts from the procedural readout pulse optimization study and provides a compact way to inspect the main frontiers and representative traces.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

study_dir = Path.cwd()
data_dir = study_dir / 'data'
summary = json.loads((data_dir / 'study_summary.json').read_text())
traces = np.load(data_dir / 'representative_traces.npz')

summary['hierarchy']

In [ ]:
durations = np.array([row['duration_ns'] for row in summary['hierarchy']], dtype=float)
ideal = np.array([row['ideal_bound'] for row in summary['hierarchy']], dtype=float)
detector = np.array([row['detector_bound'] for row in summary['hierarchy']], dtype=float)
realistic = np.array([row['realistic_best'] for row in summary['hierarchy']], dtype=float)

plt.figure(figsize=(7, 4))
plt.plot(durations, ideal, label='Ideal linear bound', linewidth=2.2)
plt.plot(durations, detector, label='Detector-limited', linewidth=2.0)
plt.plot(durations, realistic, label='Realistic balanced replay', linewidth=2.0)
plt.xlabel('Readout duration (ns)')
plt.ylabel('Assignment fidelity')
plt.ylim(0.45, 1.01)
plt.legend(frameon=False)
plt.tight_layout()

In [ ]:
family = 'procedural_segments'
t_ns = traces[f'{family}_t_ns']
alpha_g = traces[f'{family}_alpha_g']
alpha_e = traces[f'{family}_alpha_e']
signal_g = traces[f'{family}_signal_g']
signal_e = traces[f'{family}_signal_e']

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].plot(t_ns[:-1], np.abs(signal_g[:-1]), label='|s_g|')
axes[0].plot(t_ns[:-1], np.abs(signal_e[:-1]), label='|s_e|')
axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('Output amplitude')
axes[0].legend(frameon=False)

axes[1].plot(alpha_g.real, alpha_g.imag, label='alpha_g')
axes[1].plot(alpha_e.real, alpha_e.imag, label='alpha_e')
axes[1].set_xlabel('Re(alpha)')
axes[1].set_ylabel('Im(alpha)')
axes[1].legend(frameon=False)

rep = summary['representative'][family]['metrics']
mu = np.sqrt(rep['snr2_ideal'] * summary['config']['eta_nominal'])
x = np.linspace(-4, 4, 400)
gaussian = lambda center: np.exp(-0.5 * (x - center) ** 2) / np.sqrt(2 * np.pi)
axes[2].plot(x, gaussian(-mu), label='Ground')
axes[2].plot(x, gaussian(mu), label='Excited')
axes[2].axvline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_xlabel('Matched-filter score')
axes[2].set_ylabel('Probability density')
axes[2].legend(frameon=False)

fig.tight_layout()